# 03 · Comparación de modelos (desde MLflow)

**Proyecto:** Alerta temprana de deserción · SENA
**Autor:** Andrés · **Revisor:** David

**Objetivo:** justificar con datos la elección del modelo final (ADR 0002). Buscamos:

1. Leer de MLflow los runs del experimento `desercion-comparacion`. Este cuaderno **no reentrena nada**: los runs los genera `uv run python -m desercion.comparar_modelos`.
2. Comprobar que los números coinciden con los del pipeline (`reports/tuning_lightgbm.json`).
3. Comparar los modelos **fold a fold**: como todos se evaluaron en los mismos 5 folds, la comparación es pareada.
4. Decidir si la ventaja de LightGBM es real o está dentro del ruido.

Solo se usa train (validación cruzada por cohorte). Test no participa: elegir modelo mirando test sería ajustar el modelo a test.

## 1. Configuración

Las credenciales de DagsHub se leen de `.env` (no se versiona). Sin `MLFLOW_TRACKING_URI`, MLflow buscaría runs en una carpeta local vacía.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from desercion.comparar_modelos import EXPERIMENTO, N_FOLDS

# Raíz del repositorio: funciona tanto si el cuaderno se abre desde notebooks/ como desde la raíz.
RAIZ = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
load_dotenv(RAIZ / ".env")
assert os.getenv("MLFLOW_TRACKING_URI"), "Falta MLFLOW_TRACKING_URI en .env"

# Orden y nombres para tablas y gráficos.
NOMBRES = {
    "lightgbm": "LightGBM (tuneado)",
    "logistica": "Regresión logística",
    "red_neuronal": "Red neuronal",
    "azar": "Azar",
}
print("MLflow:", mlflow.get_tracking_uri())

## 2. Runs de la última comparación

Cada ejecución de `comparar_modelos` registra 4 runs con la misma etiqueta `comparacion_id`. Se toma la tanda más reciente para no mezclar resultados de versiones distintas del código.

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
ultima = runs["tags.comparacion_id"].max()
runs = runs[runs["tags.comparacion_id"] == ultima].set_index("tags.modelo")

assert set(runs.index) == set(NOMBRES), f"Faltan modelos en la tanda {ultima}"
assert runs["tags.git_sucio"].eq("False").all(), (
    "Runs con cambios sin commit: no son reproducibles"
)

print(
    f"Tanda {ultima} · commit {runs['tags.git_commit'].iloc[0][:7]} · "
    f"datos train md5 {runs['tags.datos_train_md5'].iloc[0][:8]}"
)

resumen = (
    runs[["metrics.pr_auc_media", "metrics.pr_auc_std", "metrics.roc_auc_media"]]
    .set_axis(["PR-AUC media", "PR-AUC desv.", "ROC-AUC media"], axis=1)
    .reindex(list(NOMBRES))
    .rename(index=NOMBRES)
    .rename_axis(None)
    .round(3)
)
display(resumen)

La línea `git_sucio` importa: un run con cambios sin commit no se puede reconstruir, porque no hay un commit que tenga exactamente ese código. Por eso la comparación definitiva se ejecuta **después** de hacer commit.

## 3. ¿Coincide con el pipeline?

LightGBM usa los hiperparámetros que eligió `tune.py`. Su PR-AUC media en los mismos folds debe ser la que `tune.py` escribió en `reports/tuning_lightgbm.json`. Si esta celda falla, los folds o los datos no son los mismos.

In [ ]:
tuning = json.loads(
    (RAIZ / "reports" / "tuning_lightgbm.json").read_text(encoding="utf-8")
)
pr_auc_mlflow = runs.loc["lightgbm", "metrics.pr_auc_media"]

print(f"tune.py: {tuning['pr_auc_cv_media']:.4f} · MLflow: {pr_auc_mlflow:.4f}")
assert np.isclose(pr_auc_mlflow, tuning["pr_auc_cv_media"]), "No coincide con tune.py"
print("✅ Mismos folds y mismos datos que tune.py")

## 4. Fold a fold

Cada fold valida sobre cohortes que el modelo no vio. Las métricas por fold se guardaron con `step = fold`, así que se recuperan con el historial de la métrica.

In [ ]:
cliente = mlflow.MlflowClient()


def por_fold(run_id: str, metrica: str) -> pd.Series:
    historial = cliente.get_metric_history(run_id, metrica)
    return pd.Series({m.step: m.value for m in historial}).sort_index()


pr_auc = pd.DataFrame(
    {modelo: por_fold(runs.loc[modelo, "run_id"], "pr_auc_fold") for modelo in NOMBRES}
)
pr_auc.index.name = "fold"
assert len(pr_auc) == N_FOLDS
display(pr_auc.rename(columns=NOMBRES).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for modelo, estilo in zip(NOMBRES, ["-o", "-s", "-^", "--"]):
    ax.plot(pr_auc.index, pr_auc[modelo], estilo, label=NOMBRES[modelo])
ax.set(
    xlabel="Fold (cohortes de validación)",
    ylabel="PR-AUC",
    xticks=pr_auc.index,
    title="PR-AUC por fold: los modelos suben y bajan juntos",
)
ax.legend(loc="center right")
plt.show()

Lo que se ve: las líneas de los tres modelos **se mueven juntas**. El fold 2 es difícil para todos y el 4 es fácil para todos. La variación entre folds (de ~0.50 a ~0.62) es mucho mayor que la distancia entre modelos. Por eso comparar solo las medias engaña: hay que comparar dentro de cada fold.

## 5. Comparación pareada contra la logística

Para cada fold se resta la PR-AUC de la logística. Positivo = el modelo le gana a la logística en ese fold.

In [ ]:
diferencias = pr_auc[["lightgbm", "red_neuronal"]].sub(pr_auc["logistica"], axis=0)
display(diferencias.rename(columns=NOMBRES).round(3))

pareada = pd.DataFrame(
    {
        "folds ganados": (diferencias > 0).sum(),
        "diferencia media": diferencias.mean(),
        "diferencia mín.": diferencias.min(),
        "diferencia máx.": diferencias.max(),
    }
).rename(index=NOMBRES)
display(pareada.round(3))

rango_folds = pr_auc["logistica"].max() - pr_auc["logistica"].min()
print(f"Rango de la logística entre folds: {rango_folds:.3f}")
print(f"Ventaja media de LightGBM:        {diferencias['lightgbm'].mean():.3f}")

## 6. Conclusión

- **LightGBM** le gana a la logística en 4 de 5 folds, pero por muy poco (≈0.007 de media) y pierde en uno. Esa ventaja es más de diez veces menor que la variación entre folds. Además está **sesgada a su favor**: sus hiperparámetros se eligieron con Optuna en estos mismos folds, y su PR-AUC es el mejor de 50 intentos. La logística no se tuneó.
- **La red neuronal** queda por debajo de la logística en 4 de 5 folds (≈0.016 menos de media). Con unas 8000 filas no tiene de dónde aprender relaciones más complejas.
- **Los tres modelos** superan con claridad al azar (≈0.31) y llegan al mismo techo (~0.56–0.58). El límite está en la información de los datos, no en la complejidad del modelo.

**Decisión (ADR 0002):** regresión logística. Rinde lo mismo y es más simple de mantener y desplegar. Además sus coeficientes dan directamente la explicación por aprendiz (`explain.py`), sin herramientas adicionales.

**Cómo reproducir:**
```powershell
uv run python -m desercion.comparar_modelos   # registra una tanda nueva en MLflow
# luego: Restart → Run All en este cuaderno
```